# Partial Cross-Entropy Loss for Weakly-Supervised Remote Sensing Segmentation

---

Standard segmentation requires dense pixel masks. This notebook implements **Partial Cross-Entropy (pCE)** — a custom loss that trains on sparse point annotations only. The `MASK_labeled` binary tensor zeroes out gradient at unlabeled pixels:

$$pCE = \frac{\sum FocalLoss(pred, GT) \times MASK_{labeled}}{\sum MASK_{labeled}}$$

Two experiments explore:
- **Experiment A**: How does point annotation density affect segmentation mIoU?
- **Experiment B**: How does focal loss gamma (hard example mining) interact with sparse supervision?

**Dataset**: ISPRS Potsdam (6-class urban remote sensing, free public benchmark)  
**Model**: DeepLabV3+ (ResNet-50, ImageNet pretrained)  
**Framework**: PyTorch + segmentation-models-pytorch

---

### Notebook structure
```
Cell 0  — Environment check & reproducibility seeds
Cell 1  — Imports & configuration
Cell 2A — Colab raw-zip staging & conversion
Cell 2  — Dataset download & tile extraction
Cell 3  — PotsdamDataset class + point sampler
Cell 4  — PartialCrossEntropyLoss implementation
Cell 5  — Model definition (DeepLabV3+)
Cell 6  — Training loop + metrics
Cell 7  — Experiment A: point density ablation
Cell 8  — Experiment B: focal gamma ablation
Cell 9  — Qualitative results visualization
Cell 10 — Results summary table & per-class IoU
```


In [ ]:
# ============================================================
# Cell 0 — Environment check & reproducibility
# ============================================================
import sys, torch, torchvision

print(f'Python:      {sys.version.split()[0]}')
print(f'PyTorch:     {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'CUDA:        {torch.version.cuda}')
print(f'Device:      {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU:         {torch.cuda.get_device_name(0)}')

import random, numpy as np, os
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'Seed {SEED} set for reproducibility.')


In [ ]:
# ============================================================
# Cell 1 — Imports & configuration
# ============================================================
# Uncomment if running in Colab / fresh environment:
!pip install segmentation-models-pytorch python-dotenv -q

import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import os, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from dotenv import find_dotenv, load_dotenv
import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix

# -----------------------------------------------------------
# Runtime-aware data path helpers
# -----------------------------------------------------------
ENV_FILE = find_dotenv('.env', usecwd=True)
if ENV_FILE:
    load_dotenv(ENV_FILE, override=False)

def env_or_default(key: str, default: str) -> str:
    value = os.environ.get(key)
    return value if value else default

def env_flag(key: str, default: bool) -> bool:
    value = os.environ.get(key)
    if value is None or value == '':
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'on'}

def env_int(key: str, default: int) -> int:
    value = os.environ.get(key)
    if value is None or value == '':
        return default
    return int(value)

def env_float(key: str, default: float) -> float:
    value = os.environ.get(key)
    if value is None or value == '':
        return default
    return float(value)

def env_optional_float(key: str):
    value = os.environ.get(key)
    if value is None or value == '':
        return None
    return float(value)

PROJECT_ROOT = Path(env_or_default('PROJECT_ROOT', str(Path.cwd()))).expanduser()
DEFAULT_LOCAL_DATA_ROOT = Path(
    env_or_default('POTSDAM_LOCAL_DATA_ROOT', str(PROJECT_ROOT / 'data' / 'potsdam'))
).expanduser()
DEFAULT_COLAB_STAGE_ROOT = Path(
    env_or_default('POTSDAM_COLAB_STAGE_ROOT', '/content/data')
).expanduser()
DEFAULT_COLAB_DATA_ROOT = Path(
    env_or_default('POTSDAM_COLAB_DATA_ROOT', str(DEFAULT_COLAB_STAGE_ROOT / 'potsdam'))
).expanduser()
DEFAULT_COLAB_PROCESSED_DRIVE_ROOT = Path(
    env_or_default('POTSDAM_COLAB_PROCESSED_DRIVE_ROOT', '/content/drive/MyDrive/data/potsdam')
).expanduser()
DEFAULT_COLAB_DRIVE_DATA_ROOT = Path(
    env_or_default('POTSDAM_COLAB_DRIVE_DATA_ROOT', str(DEFAULT_COLAB_PROCESSED_DRIVE_ROOT))
).expanduser()
DEFAULT_COLAB_OUTPUT_ROOT = Path(
    env_or_default('POTSDAM_COLAB_OUTPUT_ROOT', '/content/drive/MyDrive/remote-sensing-segmentation-pipeline-outputs')
).expanduser()
DEFAULT_COLAB_IMG_DIR_ZIP = Path(
    env_or_default('POTSDAM_COLAB_IMG_DIR_ZIP', str(DEFAULT_COLAB_PROCESSED_DRIVE_ROOT / 'img_dir.zip'))
).expanduser()
DEFAULT_COLAB_ANN_DIR_ZIP = Path(
    env_or_default('POTSDAM_COLAB_ANN_DIR_ZIP', str(DEFAULT_COLAB_PROCESSED_DRIVE_ROOT / 'ann_dir.zip'))
).expanduser()
DEFAULT_COLAB_RGB_ZIP = Path(
    env_or_default('POTSDAM_COLAB_RGB_ZIP', '/content/drive/MyDrive/data/dataset/2_Ortho_RGB.zip')
).expanduser()
DEFAULT_COLAB_LABEL_ZIP = Path(
    env_or_default('POTSDAM_COLAB_LABEL_ZIP', '/content/drive/MyDrive/data/dataset/5_Labels_all_noBoundary.zip')
).expanduser()
DEFAULT_CONVERTER_FALLBACK_URL = env_or_default(
    'POTSDAM_CONVERTER_FALLBACK_URL',
    os.environ.get(
        'POTSDAM_CONVERTER_SCRIPT_URL',
        'https://raw.githubusercontent.com/DavidOgalo/remote-sensing-segmentation-pipeline/main/tools/dataset_converters/custom_potsdam.py',
    ),
)

def detect_runtime() -> str:
    return 'colab' if str(Path.cwd()).startswith('/content') else 'local'

def resolve_default_data_root() -> str:
    """Choose a dataset root that works for the active kernel.

    Priority:
      1. POTSDAM_DATA_ROOT environment variable
      2. Local workspace path when running a local kernel
      3. Colab local SSD staging path when running a Colab-style kernel
      4. Relative fallback for manual override workflows
    """
    env_override = os.environ.get('POTSDAM_DATA_ROOT')
    if env_override:
        return str(Path(env_override).expanduser())

    runtime = detect_runtime()
    if runtime == 'local':
        return str(DEFAULT_LOCAL_DATA_ROOT)
    if runtime == 'colab':
        return str(DEFAULT_COLAB_DATA_ROOT)

    return './data/potsdam'

def resolve_default_output_root() -> str:
    env_override = os.environ.get('POTSDAM_OUTPUT_ROOT')
    if env_override:
        return str(Path(env_override).expanduser())

    runtime = detect_runtime()
    if runtime == 'local':
        return str(PROJECT_ROOT)
    if runtime == 'colab':
        return str(DEFAULT_COLAB_OUTPUT_ROOT)

    return str(PROJECT_ROOT)

def resolve_num_workers() -> int:
    """Notebook-safe dataloader worker count.

    Classes defined in notebook cells are not safely picklable under the
    spawn start method used by macOS/Python 3.14 multiprocessing, so default
    to 0 unless the user explicitly overrides it.
    """
    env_value = os.environ.get('DATALOADER_NUM_WORKERS')
    if env_value is not None and env_value != '':
        return max(0, int(env_value))
    return 0

def resolve_run_mode() -> str:
    run_mode = env_or_default('RUN_MODE', 'FULL_RUN').strip().upper()
    valid_modes = {'FAST_DEV_RUN', 'FULL_RUN'}
    if run_mode not in valid_modes:
        raise ValueError(f'RUN_MODE must be one of {sorted(valid_modes)}. Got: {run_mode}')
    return run_mode

def build_output_paths(output_root: Path) -> dict:
    output_root = Path(output_root).expanduser()
    return {
        'output_root': str(output_root),
        'figures_dir': str(output_root / 'figures'),
        'checkpoints_dir': str(output_root / 'checkpoints'),
        'results_dir': str(output_root / 'results'),
    }

RUN_MODE = resolve_run_mode()
IS_FAST_DEV_RUN = RUN_MODE == 'FAST_DEV_RUN'

def get_run_profile(run_mode: str) -> dict:
    if run_mode == 'FAST_DEV_RUN':
        return {
            'batch_size': 2,
            'epochs': 2,
            'point_densities': [0.01, 0.05],
            'focal_gammas': [0.0, 2.0],
            'fixed_density': 0.01,
            'max_train_tiles': 24,
            'max_val_tiles': 12,
            'early_stopping_patience': 2,
            'early_stopping_min_epochs': 1,
        }
    return {
        'batch_size': 4,
        'epochs': 30,
        'point_densities': [0.001, 0.005, 0.01, 0.05],
        'focal_gammas': [0.0, 0.5, 1.0, 2.0],
        'fixed_density': 0.01,
        'max_train_tiles': None,
        'max_val_tiles': None,
        'early_stopping_patience': 5,
        'early_stopping_min_epochs': 8,
    }

RUN_PROFILE = get_run_profile(RUN_MODE)
USE_AMP_DEFAULT = torch.cuda.is_available()
USE_AMP = env_flag('USE_AMP', USE_AMP_DEFAULT) and torch.cuda.is_available()
RESUME_TRAINING = env_flag('RESUME_TRAINING', True)
STAGE_DRIVE_DATA_TO_CONTENT = env_flag('POTSDAM_COLAB_STAGE_DRIVE_DATA_TO_CONTENT', True)
# EXPERIMENT_A_DENSITY_OVERRIDE = env_optional_float('EXPERIMENT_A_DENSITY_OVERRIDE')
# EXPERIMENT_B_GAMMA_OVERRIDE = env_optional_float('EXPERIMENT_B_GAMMA_OVERRIDE')
EXPERIMENT_A_DENSITY_OVERRIDE = None
EXPERIMENT_B_GAMMA_OVERRIDE = None
OUTPUT_PATHS = build_output_paths(Path(resolve_default_output_root()))

# -----------------------------------------------------------
# Central configuration
# -----------------------------------------------------------
CFG = {
    'runtime':                          detect_runtime(),
    'run_mode':                         RUN_MODE,
    'is_fast_dev_run':                  IS_FAST_DEV_RUN,
    'project_root':                     str(PROJECT_ROOT),
    'data_root':                        resolve_default_data_root(),
    'colab_stage_root':                 str(DEFAULT_COLAB_STAGE_ROOT),
    'colab_raw_root':                   str(DEFAULT_COLAB_STAGE_ROOT / 'dataset'),
    'colab_processed_drive_root':       str(DEFAULT_COLAB_PROCESSED_DRIVE_ROOT),
    'colab_drive_data_root':            str(DEFAULT_COLAB_DRIVE_DATA_ROOT),
    'colab_stage_drive_data_to_content': STAGE_DRIVE_DATA_TO_CONTENT,
    'colab_output_root':                str(DEFAULT_COLAB_OUTPUT_ROOT),
    'colab_img_dir_zip':                str(DEFAULT_COLAB_IMG_DIR_ZIP),
    'colab_ann_dir_zip':                str(DEFAULT_COLAB_ANN_DIR_ZIP),
    'colab_rgb_zip':                    str(DEFAULT_COLAB_RGB_ZIP),
    'colab_label_zip':                  str(DEFAULT_COLAB_LABEL_ZIP),
    'converter_fallback_url':           DEFAULT_CONVERTER_FALLBACK_URL,
    'img_size':                         512,
    'stride_size':                      256,
    'num_classes':                      6,
    'train_ratio':                      0.7,
    'backbone':                         'resnet50',
    'batch_size':                       RUN_PROFILE['batch_size'],
    'epochs':                           RUN_PROFILE['epochs'],
    'lr':                               1e-4,
    'weight_decay':                     1e-4,
    'grad_clip':                        1.0,
    'device':                           'cuda' if torch.cuda.is_available() else 'cpu',
    'use_amp':                          USE_AMP,
    'resume_training':                  RESUME_TRAINING,
    'early_stopping_patience':          env_int('EARLY_STOPPING_PATIENCE', RUN_PROFILE['early_stopping_patience']),
    'early_stopping_min_epochs':        env_int('EARLY_STOPPING_MIN_EPOCHS', RUN_PROFILE['early_stopping_min_epochs']),
    'early_stopping_delta':             env_float('EARLY_STOPPING_DELTA', 0.0),
    'train_img_dir':                    'img_dir/train',
    'train_ann_dir':                    'ann_dir/train',
    'val_img_dir':                      'img_dir/val',
    'val_ann_dir':                      'ann_dir/val',
    'num_workers':                      resolve_num_workers(),
    'pin_memory':                       torch.cuda.is_available(),
    'point_densities':                  RUN_PROFILE['point_densities'],
    'focal_gammas':                     RUN_PROFILE['focal_gammas'],
    'fixed_density':                    RUN_PROFILE['fixed_density'],
    'experiment_a_density_override':    EXPERIMENT_A_DENSITY_OVERRIDE,
    'experiment_b_gamma_override':      EXPERIMENT_B_GAMMA_OVERRIDE,
    'max_train_tiles':                  RUN_PROFILE['max_train_tiles'],
    'max_val_tiles':                    RUN_PROFILE['max_val_tiles'],
    'output_root':                      OUTPUT_PATHS['output_root'],
    'figures_dir':                      OUTPUT_PATHS['figures_dir'],
    'checkpoints_dir':                  OUTPUT_PATHS['checkpoints_dir'],
    'results_dir':                      OUTPUT_PATHS['results_dir'],
}

def ensure_output_directories() -> bool:
    output_root = Path(CFG['output_root']).expanduser()
    drive_mount_root = Path('/content/drive')
    if (
        CFG['runtime'] == 'colab'
        and str(output_root).startswith(str(drive_mount_root))
        and not drive_mount_root.exists()
    ):
        print('Drive is not mounted yet. Output directory creation will run in Cell 2A.')
        return False

    for directory in [CFG['figures_dir'], CFG['checkpoints_dir'], CFG['results_dir']]:
        os.makedirs(directory, exist_ok=True)
    return True

def apply_output_root(output_root: Path) -> Path:
    output_paths = build_output_paths(output_root)
    os.environ['POTSDAM_OUTPUT_ROOT'] = output_paths['output_root']
    CFG['output_root'] = output_paths['output_root']
    CFG['figures_dir'] = output_paths['figures_dir']
    CFG['checkpoints_dir'] = output_paths['checkpoints_dir']
    CFG['results_dir'] = output_paths['results_dir']
    ensure_output_directories()
    return Path(CFG['output_root']).expanduser()

CLASS_NAMES = [
    'Impervious surface', 'Building', 'Low vegetation',
    'Tree', 'Car', 'Clutter',
]
CLASS_COLORS_RGB = [
    (255, 255, 255),
    (0,   0,   255),
    (0,   255, 255),
    (0,   255, 0  ),
    (255, 255, 0  ),
    (255, 0,   0  ),
]

ensure_output_directories()

print('Configuration loaded.')
print(f'Runtime: {CFG["runtime"]}')
print(f'Run mode: {CFG["run_mode"]}')
print(f'Project root: {CFG["project_root"]}')
print(f'Device: {CFG["device"]}')
print(f'Data root: {CFG["data_root"]}')
print(f'Output root: {CFG["output_root"]}')
print(f'Loaded .env: {ENV_FILE if ENV_FILE else "not found"}')
print(f'DataLoader workers: {CFG["num_workers"]}')
print(f'Pin memory: {CFG["pin_memory"]}')
print(f'Use AMP: {CFG["use_amp"]}')
print(f'Resume training: {CFG["resume_training"]}')
print(f'Early stopping patience: {CFG["early_stopping_patience"]}')
print(f'Early stopping min epochs: {CFG["early_stopping_min_epochs"]}')
print(f'Experiment A density override: {CFG["experiment_a_density_override"]}')
print(f'Experiment B gamma override:   {CFG["experiment_b_gamma_override"]}')
if CFG['runtime'] == 'colab':
    print(f'Colab drive data root:       {CFG["colab_drive_data_root"]}')
    print(f'Stage Drive data to /content: {CFG["colab_stage_drive_data_to_content"]}')
    print(f'Colab output root:           {CFG["colab_output_root"]}')
    print(f'Colab processed Drive root:  {CFG["colab_processed_drive_root"]}')
    print(f'Colab img_dir.zip:           {CFG["colab_img_dir_zip"]}')
    print(f'Colab ann_dir.zip:           {CFG["colab_ann_dir_zip"]}')
    print(f'Colab RGB zip:               {CFG["colab_rgb_zip"]}')
    print(f'Colab label zip:             {CFG["colab_label_zip"]}')
    print(f'Colab stage root:            {CFG["colab_stage_root"]}')
    print(f'Converter fallback URL:      {CFG["converter_fallback_url"]}')
print(f'Batch size: {CFG["batch_size"]}')
print(f'Epochs:     {CFG["epochs"]}')
print(f'Exp A densities: {CFG["point_densities"]}')
print(f'Exp B gammas:    {CFG["focal_gammas"]}')
print(f'Max train tiles: {CFG["max_train_tiles"]}')
print(f'Max val tiles:   {CFG["max_val_tiles"]}')
print(f'Checkpoints dir: {CFG["checkpoints_dir"]}')
print(f'Figures dir:     {CFG["figures_dir"]}')
print(f'Results dir:     {CFG["results_dir"]}')

## Dataset Setup

### Dataset Choice
ISPRS Potsdam or Vaihingen are both good fits for weakly-supervised remote sensing segmentation because they provide dense semantic labels that can be converted into sparse point supervision during training.

- Potsdam: 38 patches, 6000×6000 px each, 6 classes (impervious surface, building, low vegetation, tree, car, clutter), RGB + IR
- Vaihingen: 33 patches, varying sizes, same 6 classes, IR-R-G channels

### Option A — ISPRS Potsdam workflow used in this repository
#### Data Source — Urban Modelling and Semantic Labelling Benchmark (UrbanSemLab)
Full reference data with labels:
- Main benchmark overview: https://www.isprs.org/resources/datasets/benchmarks/UrbanSemLab/
- Potsdam-specific page: https://www.isprs.org/resources/datasets/benchmarks/UrbanSemLab/2d-sem-label-potsdam.aspx

#### Official Download
1. Go to the Potsdam seafile folder:
https://seafile.projekt.uni-hannover.de/f/429be50cc79d423ab6c4/
2. Password: CjwcipT4-P8g
3. Download the main archives, including:
   - `2_Ortho_RGB.zip`
   - `5_Labels_all_noBoundary.zip`

**Note**: The complete Potsdam download is large (about 13.3 GB). After extraction you get the original 38 large orthophoto patches and their label TIFFs.

### Actual preparation workflow used here
In this project, the raw Potsdam zip files were downloaded and placed in `data/dataset/`, then converted with a standalone script that reproduces the official MMSegmentation cropping logic without depending on `mmcv` or `mmengine`. The relevant files are:

- `tools/dataset_converters/mmsegmentation_potsdam.py` — reference copy of the original OpenMMLab-style converter
- `tools/dataset_converters/custom_potsdam.py` — Python 3.14-compatible replacement used for this repository

The standalone script was necessary because the upstream `mmcv` build path is currently incompatible with this Python 3.14 environment: prebuilt wheels are not available, source builds fall back to legacy packaging paths, and newer `setuptools` versions removed `pkg_resources`, which breaks the install path the original script depends on.

Run the converter with:

```bash
python tools/dataset_converters/custom_potsdam.py data/dataset --out_dir data/potsdam --clip_size 512 --stride_size 256
```

This produces the cropped dataset in MMSegmentation-style folders:

```text
data/potsdam/
├── img_dir/
│   ├── train/
│   └── val/
└── ann_dir/
    ├── train/
    └── val/
```

### Runtime-aware path behavior
The notebook supports both local VS Code execution and Colab-style `/content` execution. Dataset path selection is driven by `.env` variables loaded in Cell 1.

Default behavior:
- Local kernel: prefer `POTSDAM_LOCAL_DATA_ROOT` or `PROJECT_ROOT/data/potsdam` and keep outputs in the repository root
- Colab kernel: prefer `/content/data/potsdam` after running the staging cell, so training reads from Colab local SSD rather than the Drive mount
- Colab outputs: default to a Drive-backed root so checkpoints, figures, and JSON summaries persist across disconnects
- Explicit override for any runtime: set `POTSDAM_DATA_ROOT` for inputs or `POTSDAM_OUTPUT_ROOT` for outputs

Recommended `.env` variables:

```env
PROJECT_ROOT=/path/to/remote-sensing-segmentation-pipeline
POTSDAM_LOCAL_DATA_ROOT=/path/to/remote-sensing-segmentation-pipeline/data/potsdam
POTSDAM_COLAB_STAGE_ROOT=/content/data
POTSDAM_COLAB_DATA_ROOT=/content/data/potsdam
POTSDAM_COLAB_PROCESSED_DRIVE_ROOT=/content/drive/MyDrive/data/potsdam
POTSDAM_COLAB_DRIVE_DATA_ROOT=/content/drive/MyDrive/data/potsdam
POTSDAM_COLAB_OUTPUT_ROOT=/content/drive/MyDrive/remote-sensing-segmentation-pipeline-outputs
POTSDAM_COLAB_IMG_DIR_ZIP=/content/drive/MyDrive/data/potsdam/img_dir.zip
POTSDAM_COLAB_ANN_DIR_ZIP=/content/drive/MyDrive/data/potsdam/ann_dir.zip
POTSDAM_COLAB_RGB_ZIP=/content/drive/MyDrive/data/dataset/2_Ortho_RGB.zip
POTSDAM_COLAB_LABEL_ZIP=/content/drive/MyDrive/data/dataset/5_Labels_all_noBoundary.zip
# Optional path to the converter script inside the Colab filesystem
# POTSDAM_CONVERTER_SCRIPT=/content/remote-sensing-segmentation-pipeline/tools/dataset_converters/custom_potsdam.py
# Optional hard override for either runtime
# POTSDAM_DATA_ROOT=/absolute/path/to/data/potsdam
# POTSDAM_OUTPUT_ROOT=/absolute/path/to/output/root
```

This is purposely for reproducibility, allowing the same notebook to run under different kernels by making the data root and output root environment-aware:
- Prefer a local path when running in a local kernel.
- In Colab, either use an expanded `img_dir/ann_dir` folder on Drive, extract processed `img_dir.zip` and `ann_dir.zip` into `/content/data/potsdam`, or stage the raw benchmark zips and convert them there.
- Keep training data on `/content` for throughput, but persist checkpoints, figures, and experiment summaries on Drive.
- Keep one override point in config so you can switch explicitly when needed.

### Preferred Colab workflow used by this notebook
1. Upload either an expanded Potsdam folder (`img_dir/ann_dir`), the processed `img_dir.zip` and `ann_dir.zip` archives, or the original Potsdam raw zips to Google Drive.
2. Mount Drive in Colab.
3. Run the staging cell below.
4. If an expanded Drive folder exists at `POTSDAM_COLAB_DRIVE_DATA_ROOT`, the notebook mirrors it into `/content/data/potsdam` by default so training avoids Drive I/O bottlenecks.
5. Otherwise, if processed `img_dir.zip` and `ann_dir.zip` are available, the notebook extracts them into `/content/data/potsdam` and skips raw conversion.
6. Otherwise, the notebook falls back to copying the raw Potsdam zips into `/content/data/dataset` and runs the converter in Colab so the cropped tiles are written to `/content/data/potsdam`.
7. Checkpoints, results, and figures are written to the Drive-backed output root so they survive runtime resets while training still reads from `/content`.

This avoids manually uploading the expanded `img_dir/ann_dir` tree when archives are available, preserves experiment artifacts on Drive, and still keeps the hot training path on Colab local SSD.

### Local VS Code vs Google Colab
- Local kernel: use this when the dataset lives on your machine and you want direct access to workspace files under your local filesystem
- Colab kernel: use this when training needs hosted GPU compute; keep expanded folders, processed archives, or raw zips on Drive, let the notebook mirror data into `/content`, and let outputs persist on Drive automatically

### Why dense labels are still useful here
Even though the training objective is weak supervision, the notebook starts from dense Potsdam labels and then simulates MeritiInc-style sparse supervision by sampling point annotations from each segmentation mask. That lets the project study Partial Cross-Entropy under controlled sparsity while still evaluating against full ground truth.

### Vaihingen alternative
If you want a smaller ISPRS benchmark, use the Vaihingen dataset from the same benchmark page and follow the same download-and-crop pattern. The seafile link is: https://seafile.projekt.uni-hannover.de/f/6a06a837b1f349cfa749/

### Option B — Synthetic data
Set `USE_SYNTHETIC = True` in the dataset preparation code cell below if you only want to validate the pipeline quickly without downloading Potsdam. Results from synthetic data are for smoke testing only.

### Option C — Google Colab
Mount Drive, make sure either an expanded Potsdam folder, processed `img_dir.zip` and `ann_dir.zip`, or the raw Potsdam zip files exist at your configured Drive paths, run the staging cell, and let the dataset preparation cell load the converted tiles from `/content` while outputs are written to the Drive-backed output root.

In [ ]:
# ============================================================
# Cell 2A — Colab raw-zip staging & conversion
# ============================================================
# Purpose: when running on a hosted Colab kernel, prefer the fastest reproducible
# data source available in this order: staged /content tiles, Drive tiles mirrored
# into /content, Drive-hosted processed archives, then raw Potsdam zips + converter.
# Outputs remain Drive-backed so checkpoints, figures, and summaries survive resets.


import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

def ensure_drive_mounted():
    if CFG['runtime'] != 'colab':
        return
    if Path('/content/drive').exists():
        return
    from google.colab import drive
    drive.mount('/content/drive')


def prepared_dataset_exists(root: Path) -> bool:
    required_dirs = [
        root / CFG['train_img_dir'],
        root / CFG['train_ann_dir'],
        root / CFG['val_img_dir'],
        root / CFG['val_ann_dir'],
    ]
    return all(path.exists() and any(path.iterdir()) for path in required_dirs)


def processed_archives_exist() -> tuple[bool, Path, Path]:
    img_zip = Path(CFG['colab_img_dir_zip']).expanduser()
    ann_zip = Path(CFG['colab_ann_dir_zip']).expanduser()
    return img_zip.exists() and ann_zip.exists(), img_zip, ann_zip


def extract_processed_archives(processed_root: Path, img_zip: Path, ann_zip: Path, force_rebuild=False):
    if prepared_dataset_exists(processed_root) and not force_rebuild:
        print(f'Prepared Potsdam tiles already exist at {processed_root}.')
        return
    if force_rebuild and processed_root.exists():
        shutil.rmtree(processed_root)
    processed_root.mkdir(parents=True, exist_ok=True)
    print(f'Extracting processed archive: {img_zip} -> {processed_root}')
    with zipfile.ZipFile(img_zip) as zf:
        zf.extractall(processed_root)
    print(f'Extracting processed archive: {ann_zip} -> {processed_root}')
    with zipfile.ZipFile(ann_zip) as zf:
        zf.extractall(processed_root)


def sync_drive_dataset_to_content(drive_root: Path, content_root: Path, force_rebuild=False):
    if prepared_dataset_exists(content_root) and not force_rebuild:
        print(f'Prepared Potsdam tiles already exist at {content_root}.')
        return
    if force_rebuild and content_root.exists():
        shutil.rmtree(content_root)
    content_root.mkdir(parents=True, exist_ok=True)

    rsync_bin = shutil.which('rsync')
    if rsync_bin:
        command = [rsync_bin, '-a', '--delete', f'{drive_root}/', f'{content_root}/']
        print('Mirroring expanded Drive dataset into /content...')
        print(' '.join(command))
        result = subprocess.run(command, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            if result.stderr:
                print(result.stderr)
            raise RuntimeError(f'rsync failed with exit code {result.returncode}.')
        return

    print('rsync not available. Falling back to shutil.copytree for img_dir/ann_dir.')
    for folder_name in ['img_dir', 'ann_dir']:
        source_dir = drive_root / folder_name
        target_dir = content_root / folder_name
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(source_dir, target_dir)


def resolve_converter_script_path() -> tuple[Path, str]:
    candidates = []

    env_override = os.environ.get('POTSDAM_CONVERTER_SCRIPT')
    if env_override:
        candidates.append(Path(env_override).expanduser())

    candidates.extend([
        Path.cwd() / 'tools' / 'dataset_converters' / 'custom_potsdam.py',
        Path(CFG['project_root']) / 'tools' / 'dataset_converters' / 'custom_potsdam.py',
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve(), 'local file'

    download_target = Path(CFG['colab_stage_root']) / 'tools' / 'dataset_converters' / 'custom_potsdam.py'
    download_target.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading converter script -> {download_target}')
    urlretrieve(CFG['converter_fallback_url'], download_target)
    if download_target.exists():
        return download_target.resolve(), 'downloaded fallback'

    raise FileNotFoundError(
        'Could not find or download tools/dataset_converters/custom_potsdam.py. '
        'Clone or mount the repository into Colab, set POTSDAM_CONVERTER_SCRIPT, '
        'or set POTSDAM_CONVERTER_FALLBACK_URL to a reachable raw file URL.'
    )


def run_converter(converter_path: Path, raw_root: Path, processed_root: Path):
    command = [
        sys.executable,
        str(converter_path),
        str(raw_root),
        '--out_dir', str(processed_root),
        '--clip_size', str(CFG['img_size']),
        '--stride_size', str(CFG['stride_size']),
    ]
    print('Running converter in /content...')
    print(' '.join(command))
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f'custom_potsdam.py failed with exit code {result.returncode}.')


def activate_data_root(root: Path, source_label: str) -> Path:
    os.environ['POTSDAM_DATA_ROOT'] = str(root)
    CFG['data_root'] = str(root)
    print(f'Dataset source: {source_label}')
    print(f'Active data root: {CFG["data_root"]}')
    return root


def activate_output_root(root: Path, source_label: str) -> Path:
    apply_output_root(root)
    print(f'Output source: {source_label}')
    print(f'Active output root: {CFG["output_root"]}')
    print(f'Checkpoints dir: {CFG["checkpoints_dir"]}')
    print(f'Figures dir:     {CFG["figures_dir"]}')
    print(f'Results dir:     {CFG["results_dir"]}')
    return Path(CFG['output_root']).expanduser()


def stage_and_convert_in_colab(force_rebuild=False):
    if CFG['runtime'] != 'colab':
        print('Local runtime detected. Skipping Colab staging cell.')
        return Path(CFG['data_root'])

    ensure_drive_mounted()
    output_root = Path(CFG['output_root']).expanduser()
    output_source = 'configured output directory'
    if str(output_root).startswith('/content/drive'):
        output_source = 'Drive-backed output directory'
    activate_output_root(output_root, output_source)

    stage_root = Path(CFG['colab_stage_root']).expanduser()
    raw_root = Path(CFG['colab_raw_root']).expanduser()
    processed_root = Path(CFG['data_root']).expanduser()
    drive_data_root = Path(CFG['colab_drive_data_root']).expanduser()
    stage_root.mkdir(parents=True, exist_ok=True)
    raw_root.mkdir(parents=True, exist_ok=True)
    processed_root.parent.mkdir(parents=True, exist_ok=True)

    if prepared_dataset_exists(processed_root) and not force_rebuild:
        print(f'Prepared Potsdam tiles already exist at {processed_root}.')
        return activate_data_root(processed_root, 'existing extracted tiles under /content')

    if prepared_dataset_exists(drive_data_root):
        print(f'Detected expanded Potsdam dataset on Drive: {drive_data_root}')
        if CFG['colab_stage_drive_data_to_content']:
            sync_drive_dataset_to_content(drive_data_root, processed_root, force_rebuild=force_rebuild)
            return activate_data_root(processed_root, 'Drive folder mirrored to /content')
        print('Training directly from the Drive mount is supported, but can be slower than /content.')
        return activate_data_root(drive_data_root, 'direct Drive folder')

    processed_archives_available, img_dir_zip, ann_dir_zip = processed_archives_exist()
    if processed_archives_available:
        print('Detected processed dataset archives on Drive.')
        print(f'img_dir.zip: {img_dir_zip}')
        print(f'ann_dir.zip: {ann_dir_zip}')
        extract_processed_archives(processed_root, img_dir_zip, ann_dir_zip, force_rebuild=force_rebuild)
        return activate_data_root(processed_root, 'processed archives')

    rgb_zip = Path(CFG['colab_rgb_zip']).expanduser()
    label_zip = Path(CFG['colab_label_zip']).expanduser()
    missing = [str(path) for path in [rgb_zip, label_zip] if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'Could not find an expanded Drive dataset, processed dataset archives, or raw Potsdam zip(s) on Google Drive. '
            'Provide img_dir/ann_dir under POTSDAM_COLAB_DRIVE_DATA_ROOT, '
            'or provide img_dir.zip and ann_dir.zip under POTSDAM_COLAB_PROCESSED_DRIVE_ROOT, '
            'or update POTSDAM_COLAB_RGB_ZIP and POTSDAM_COLAB_LABEL_ZIP. Missing raw files: ' + ', '.join(missing)
        )

    for source_path in [rgb_zip, label_zip]:
        target_path = raw_root / source_path.name
        if target_path.exists() and target_path.stat().st_size == source_path.stat().st_size:
            print(f'Using staged zip: {target_path}')
        else:
            print(f'Copying {source_path.name} -> {target_path}')
            shutil.copy2(source_path, target_path)

    converter_path, converter_source = resolve_converter_script_path()
    print(f'Using converter script: {converter_path}')
    print(f'Converter source: {converter_source}')
    print('Dataset source: raw Potsdam archives -> converter -> /content')
    run_converter(converter_path, raw_root, processed_root)
    return activate_data_root(processed_root, 'raw Potsdam archives -> converter')


stage_and_convert_in_colab(force_rebuild=False)

In [ ]:
# ============================================================
# Cell 2 — Dataset preparation (Data loading & tile extraction)
# ============================================================
# Canonical loader for the cropped Potsdam layout produced by
# tools/dataset_converters/custom_potsdam.py:
#   data/potsdam/
#   ├── img_dir/train/
#   ├── img_dir/val/
#   ├── ann_dir/train/
#   └── ann_dir/val/

USE_SYNTHETIC = False   # Keep False for the real Potsdam tiles

def resolve_data_root(root):
    """Resolve a dataset path using the runtime-aware configuration from Cell 1."""
    root_path = Path(root).expanduser()
    candidates = []

    if root_path.is_absolute():
        candidates.append(root_path)
    else:
        candidates.append((Path.cwd() / root_path).resolve())
        candidates.append((PROJECT_ROOT / root_path).resolve())
        candidates.append((DEFAULT_COLAB_STAGE_ROOT / root_path).resolve())

    runtime_specific = [Path(CFG['data_root']).expanduser()]
    if CFG['runtime'] == 'colab':
        runtime_specific.append(Path(CFG['colab_drive_data_root']).expanduser())
    for candidate in runtime_specific + candidates:
        if candidate.exists():
            return candidate

    return (runtime_specific + candidates)[-1]


def maybe_limit_pairs(pairs, limit, split_name):
    if limit is None or len(pairs) <= limit:
        return pairs
    print(f'[{CFG["run_mode"]}] Limiting {split_name} tiles: {len(pairs)} -> {limit}')
    return list(pairs[:limit])


def rgb_to_class(rgb_mask: np.ndarray) -> np.ndarray:
    """Convert Potsdam RGB annotation mask to integer class ids."""
    class_map = np.full(rgb_mask.shape[:2], fill_value=255, dtype=np.int64)
    for cls_idx, color in enumerate(CLASS_COLORS_RGB):
        match = np.all(rgb_mask == np.array(color), axis=-1)
        class_map[match] = cls_idx
    return class_map


def make_synthetic_dataset(root, n_tiles=30, tile_size=512, n_classes=6):
    """Generate random flat-folder tiles for pipeline smoke testing only."""
    root = resolve_data_root(root)
    img_dir = root / 'images'
    mask_dir = root / 'masks'
    img_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)
    if len(list(img_dir.glob('*.png'))) >= n_tiles:
        print('Synthetic dataset already exists. Skipping.')
        return
    print(f'Generating {n_tiles} synthetic {tile_size}x{tile_size} tiles...')
    for i in range(n_tiles):
        img = np.random.randint(0, 255, (tile_size, tile_size, 3), dtype=np.uint8)
        Image.fromarray(img).save(img_dir / f'tile_{i:04d}.png')
        cls_idx = np.random.randint(0, n_classes, (tile_size, tile_size), dtype=np.uint8)
        Image.fromarray(cls_idx).save(mask_dir / f'tile_{i:04d}.png')
    print('Done.')


def get_flat_file_paths(root):
    """Pair flat-folder synthetic images and masks."""
    root = resolve_data_root(root)
    img_dir = root / 'images'
    mask_dir = root / 'masks'
    img_paths = sorted(img_dir.glob('*.png')) + sorted(img_dir.glob('*.tif'))
    pairs = [
        (str(img_path), str(mask_dir / img_path.name))
        for img_path in img_paths
        if (mask_dir / img_path.name).exists()
    ]
    print(f'Found {len(pairs)} image/mask pairs in {root}')
    return pairs


def get_split_file_paths(root, img_subdir, ann_subdir):
    """Pair cropped Potsdam tiles from MMSegmentation-style split folders."""
    root = resolve_data_root(root)
    img_dir = root / img_subdir
    mask_dir = root / ann_subdir
    img_paths = sorted(img_dir.glob('*.png')) + sorted(img_dir.glob('*.tif'))
    pairs = []
    for img_path in img_paths:
        mask_path = mask_dir / img_path.name
        if mask_path.exists():
            pairs.append((str(img_path), str(mask_path)))
    return pairs


def explain_missing_data_root(resolved_root):
    print('\nNo Potsdam tiles found in the expected folders.')
    print(f'Kernel cwd: {Path.cwd()}')
    print(f'Runtime: {CFG["runtime"]}')
    print(f'Run mode: {CFG["run_mode"]}')
    print(f'Resolved data root: {resolved_root}')

    if CFG['runtime'] == 'colab':
        print('\nThis kernel is running under /content.')
        print('Run the staging cell first so the notebook can select an expanded Drive folder, processed archives, or raw-zips conversion path.')
    else:
        print('\nThis kernel is running locally.')
        print('Use the local workspace dataset or set POTSDAM_DATA_ROOT explicitly if your tiles live elsewhere.')

    print('Expected folders:')
    print('  img_dir/train, img_dir/val')
    print('  ann_dir/train, ann_dir/val')


def load_dataset_splits(root, use_synthetic, train_ratio=0.7, seed=42):
    """Load either synthetic flat-folder data or cropped Potsdam train/val tiles."""
    resolved_root = resolve_data_root(root)

    if use_synthetic:
        make_synthetic_dataset(resolved_root)
        pairs = get_flat_file_paths(resolved_root)
        train_pairs, val_pairs = train_val_split(pairs, train_ratio, seed)
        train_pairs = maybe_limit_pairs(train_pairs, CFG['max_train_tiles'], 'train')
        val_pairs = maybe_limit_pairs(val_pairs, CFG['max_val_tiles'], 'val')
        return train_pairs, val_pairs

    train_pairs = get_split_file_paths(resolved_root, CFG['train_img_dir'], CFG['train_ann_dir'])
    val_pairs = get_split_file_paths(resolved_root, CFG['val_img_dir'], CFG['val_ann_dir'])
    train_pairs = maybe_limit_pairs(train_pairs, CFG['max_train_tiles'], 'train')
    val_pairs = maybe_limit_pairs(val_pairs, CFG['max_val_tiles'], 'val')

    print(f'Resolved data root: {resolved_root}')
    print(f"Found {len(train_pairs)} train tiles and {len(val_pairs)} validation tiles.")
    print(f"Train images: {resolved_root / CFG['train_img_dir']}")
    print(f"Train masks:  {resolved_root / CFG['train_ann_dir']}")
    print(f"Val images:   {resolved_root / CFG['val_img_dir']}")
    print(f"Val masks:    {resolved_root / CFG['val_ann_dir']}")

    if len(train_pairs) == 0 or len(val_pairs) == 0:
        explain_missing_data_root(resolved_root)

    return train_pairs, val_pairs


def train_val_split(pairs, train_ratio=0.7, seed=42):
    rng = random.Random(seed)
    pairs = list(pairs)
    rng.shuffle(pairs)
    n = max(1, int(len(pairs) * train_ratio))
    return pairs[:n], pairs[n:]


train_pairs, val_pairs = load_dataset_splits(
    CFG['data_root'], USE_SYNTHETIC, CFG['train_ratio'], SEED
)
train_img_paths = [pair[0] for pair in train_pairs]
train_mask_paths = [pair[1] for pair in train_pairs]
val_img_paths = [pair[0] for pair in val_pairs]
val_mask_paths = [pair[1] for pair in val_pairs]

print(f'Final split -> Train: {len(train_pairs):,} tiles | Val: {len(val_pairs):,} tiles')

In [ ]:
# ============================================================
# Cell 3 - PotsdamDataset + stratified point sampler
# ============================================================

class PotsdamDataset(Dataset):
    """
    ISPRS Potsdam remote sensing segmentation dataset.
    Returns (image, full_mask, point_mask) per item.
    Point mask simulates sparse annotation via stratified sampling.
    """
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, image_paths, mask_paths, img_size=512,
                 point_density=0.01, augment=False, ignore_index=255):
        assert len(image_paths) == len(mask_paths)
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.point_density = point_density
        self.augment = augment
        self.ignore_index = ignore_index
        self.normalize = T.Normalize(mean=self.MEAN, std=self.STD)

    def __len__(self):
        return len(self.image_paths)

    def _load_mask(self, mask_path):
        mask = np.array(Image.open(mask_path))
        if mask.ndim == 3 and mask.shape[2] == 4:
            mask = mask[:, :, :3]
        return mask

    def _load_and_crop(self, img_path, mask_path):
        img = np.array(Image.open(img_path).convert('RGB'))
        mask = self._load_mask(mask_path)
        height, width = img.shape[:2]
        size = self.img_size

        if height < size or width < size:
            pad_h = max(0, size - height)
            pad_w = max(0, size - width)
            img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
            if mask.ndim == 2:
                mask = np.pad(mask, ((0, pad_h), (0, pad_w)), mode='edge')
            else:
                mask = np.pad(mask, ((0, pad_h), (0, pad_w), (0, 0)), mode='edge')
            height, width = img.shape[:2]

        row = random.randint(0, height - size) if self.augment else (height - size) // 2
        col = random.randint(0, width - size) if self.augment else (width - size) // 2

        if mask.ndim == 2:
            return img[row:row + size, col:col + size], mask[row:row + size, col:col + size]
        return img[row:row + size, col:col + size], mask[row:row + size, col:col + size, :]

    def _augment(self, img, mask):
        if random.random() > 0.5:
            img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
        if random.random() > 0.5:
            img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
        return img, mask

    def _to_class_mask(self, mask_np: np.ndarray) -> np.ndarray:
        if mask_np.ndim == 2:
            class_mask = mask_np.astype(np.int64)
        else:
            class_mask = rgb_to_class(mask_np)

        class_mask = class_mask.copy()
        invalid = (class_mask < 0) | (class_mask >= CFG['num_classes'])
        class_mask[invalid] = self.ignore_index
        return class_mask

    def sample_points(self, class_mask: np.ndarray) -> np.ndarray:
        """
        Stratified random point sampling over valid classes only.
        Guarantees at least one sample per class present in the tile.
        """
        height, width = class_mask.shape
        valid_pixels = class_mask != self.ignore_index
        valid_classes = np.unique(class_mask[valid_pixels])

        mask = np.zeros((height, width), dtype=np.float32)
        if len(valid_classes) == 0:
            return mask

        n_total = max(1, int(valid_pixels.sum() * self.point_density))
        n_per_class = max(1, n_total // len(valid_classes))

        for cls in valid_classes:
            pixels = np.argwhere(class_mask == cls)
            if len(pixels) == 0:
                continue
            n_select = min(n_per_class, len(pixels))
            chosen = pixels[np.random.choice(len(pixels), n_select, replace=False)]
            mask[chosen[:, 0], chosen[:, 1]] = 1.0

        return mask

    def __getitem__(self, idx):
        img_np, mask_np = self._load_and_crop(
            self.image_paths[idx], self.mask_paths[idx]
        )
        if self.augment:
            img_np, mask_np = self._augment(img_np, mask_np)

        class_mask = self._to_class_mask(mask_np)
        point_mask = self.sample_points(class_mask)
        img_t = torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)
        return (
            img_t,
            torch.from_numpy(class_mask).long(),
            torch.from_numpy(point_mask).float()
        )

# Smoke test
_smoke_count = min(2, len(train_img_paths), len(train_mask_paths))
if _smoke_count == 0:
    print('Smoke test skipped: no training tiles are loaded yet.')
else:
    _ds = PotsdamDataset(
        train_img_paths[:_smoke_count], train_mask_paths[:_smoke_count],
        img_size=CFG['img_size'], point_density=0.01
    )
    _img, _mask, _pm = _ds[0]
    valid_classes = sorted([int(v) for v in _mask.unique().tolist() if int(v) != 255])
    print(f'Image:      {_img.shape}  {_img.dtype}')
    print(f'Mask:       {_mask.shape}  {_mask.dtype}  classes={valid_classes}')
    print(f'Point mask: {_pm.shape}  labeled={int(_pm.sum())} px ({_pm.mean()*100:.2f}%)')

## Cell 4 — Partial Cross-Entropy Loss (core deliverable)

The formula from the MeritiInc brief:

$$pCE = \frac{\sum FocalLoss(pred, GT) \times MASK_{labeled}}{\sum MASK_{labeled}}$$

Three design properties:
1. **Gradient isolation** — `pixel_loss × point_mask` zeroes gradient at unlabeled pixels
2. **Magnitude stability** — division by `Σ(MASK_labeled)` keeps loss scale constant regardless of sparsity
3. **Hard example focus** — focal weight `(1-pt)^γ` concentrates the sparse gradient budget on hard pixels


In [ ]:
# ============================================================
# Cell 4 — PartialCrossEntropyLoss
# ============================================================

class PartialCrossEntropyLoss(nn.Module):
    """
    Partial Cross-Entropy Loss for weakly-supervised semantic segmentation.

    Trains on sparse point annotations by computing loss ONLY at labeled pixels.
    Unlabeled pixels multiply by zero — contributing no gradient.

    Formula (MeritiInc brief):
        pCE = Σ(FocalLoss(pred, GT) × MASK_labeled) / Σ(MASK_labeled)

    Args:
        gamma (float):       Focal focusing parameter >= 0. 0 = standard CE.
        ignore_index (int):  Class index to exclude (e.g. boundary void pixels).
    """
    def __init__(self, gamma: float = 2.0, ignore_index: int = 255):
        super().__init__()
        self.gamma        = gamma
        self.ignore_index = ignore_index

    def focal_loss_per_pixel(self, logits, targets):
        """
        Element-wise focal loss (no reduction).

        Steps:
          1. CE per pixel = -log(p_t)
          2. p_t = exp(-CE) = model probability of the correct class
          3. focal_weight = (1 - p_t)^gamma
             -- p_t near 1 (easy) --> weight near 0 (down-weighted)
             -- p_t near 0 (hard) --> weight near 1 (full signal)
          4. focal = focal_weight * CE
        """
        # Step 1: per-pixel CE, shape (B, H, W)
        ce = F.cross_entropy(
            logits, targets,
            ignore_index=self.ignore_index,
            reduction='none'
        )
        if self.gamma == 0.0:
            return ce  # Exact standard CE; skip exp() for speed

        # Steps 2-4: focal weighting
        pt           = torch.exp(-ce)               # probability of correct class
        focal_weight = (1.0 - pt) ** self.gamma
        return focal_weight * ce                    # (B, H, W)

    def forward(self, logits, targets, point_mask):
        """
        Args:
            logits:     (B, C, H, W) raw model outputs (pre-softmax)
            targets:    (B, H, W)    integer ground-truth labels [0, C-1]
            point_mask: (B, H, W)    float32 binary; 1=labeled, 0=unlabeled
        Returns:
            Scalar differentiable loss
        """
        pixel_loss  = self.focal_loss_per_pixel(logits, targets)  # (B, H, W)
        masked_loss = pixel_loss * point_mask                      # zeros at unlabeled
        n_labeled   = point_mask.sum().clamp(min=1.0)             # safe denominator
        return masked_loss.sum() / n_labeled

    def __repr__(self):
        return f'PartialCrossEntropyLoss(gamma={self.gamma})'


# ---- Mathematical validation ----
print('Validating PartialCrossEntropyLoss...')
_B, _C, _H, _W = 2, 6, 64, 64
torch.manual_seed(42)
_lg = torch.randn(_B, _C, _H, _W)
_tg = torch.randint(0, _C, (_B, _H, _W))
_dn = torch.ones(_B, _H, _W)

# 1. gamma=0 with dense mask == standard CE
_pce0 = PartialCrossEntropyLoss(gamma=0.0)(_lg, _tg, _dn)
_sce  = F.cross_entropy(_lg, _tg)
print(f'  gamma=0 dense pCE: {_pce0.item():.5f}')
print(f'  Standard CE:       {_sce.item():.5f}')
print(f'  Match: {torch.allclose(_pce0, _sce, atol=1e-4)}')

# 2. Focal loss < CE for easy (confident) examples
_el = torch.full((_B,_C,_H,_W), -10.0); _el[:,0] = 10.0
_et = torch.zeros(_B,_H,_W, dtype=torch.long)
_ce_easy = PartialCrossEntropyLoss(gamma=0.0)(_el, _et, _dn).item()
_fl_easy = PartialCrossEntropyLoss(gamma=2.0)(_el, _et, _dn).item()
print(f'  Easy CE: {_ce_easy:.6f}  Focal(g=2): {_fl_easy:.6f}  focal<CE: {_fl_easy < _ce_easy}')

# 3. Empty mask -> loss = 0 (not NaN)
_em_loss = PartialCrossEntropyLoss(gamma=2.0)(_lg, _tg, torch.zeros(_B,_H,_W))
print(f'  Empty mask loss: {_em_loss.item():.4f} (should be 0.0, not NaN)')
print('All validations passed.')


In [ ]:
# ============================================================
# Cell 5 — Model: Model setup (DeepLabV3+ with pretrained backbone, ResNet-50)
# ============================================================

def build_model(num_classes=6, backbone='resnet50'):
    """
    DeepLabV3+ with ImageNet-pretrained ResNet-50 encoder.

    Rationale:
    - ASPP (Atrous Spatial Pyramid Pooling) captures multi-scale context:
      essential when objects range from cars (~10px) to buildings (~500px).
    - ImageNet pretraining compensates for sparse gradient signal of pCE.
    - activation=None returns raw logits; F.cross_entropy handles softmax
      internally for numerical stability.
    """
    return smp.DeepLabV3Plus(
        encoder_name    = backbone,
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = num_classes,
        activation      = None,
    )

_m = build_model(CFG['num_classes'], CFG['backbone'])
_m.eval()  # Smoke test uses batch size 1, so avoid BatchNorm training-mode constraints.
_x = torch.randn(1, 3, CFG['img_size'], CFG['img_size'])
with torch.no_grad():
    _o = _m(_x)
print(f'Input:  {_x.shape}')
print(f'Output: {_o.shape}  (B, C, H, W logits)')
total = sum(p.numel() for p in _m.parameters())
train = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {train:,}')
del _m, _x, _o

In [ ]:
# ============================================================
# Cell 6 — Training loop + metrics
# ============================================================

def compute_metrics(preds, targets, num_classes):
    """
    Compute mIoU and pixel accuracy using full ground-truth masks.
    Evaluation always uses complete masks regardless of training sparsity.
    """
    preds_np   = preds.cpu().numpy().flatten().astype(np.int64)
    targets_np = targets.cpu().numpy().flatten().astype(np.int64)
    cm         = confusion_matrix(targets_np, preds_np, labels=list(range(num_classes)))
    inter      = np.diag(cm)
    union      = cm.sum(1) + cm.sum(0) - inter
    iou        = np.where(union > 0, inter / union, np.nan)
    return {
        'miou':          float(np.nanmean(iou)),
        'pixel_acc':     float(inter.sum() / (cm.sum() + 1e-8)),
        'iou_per_class': iou.tolist(),
    }


def checkpoint_model_state(checkpoint):
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        return checkpoint['model_state_dict']
    return checkpoint


def make_grad_scaler(enabled):
    amp_ns = getattr(torch, 'amp', None)
    if amp_ns is not None and hasattr(amp_ns, 'GradScaler'):
        return amp_ns.GradScaler('cuda', enabled=enabled)
    return getattr(torch.cuda, 'amp').GradScaler(enabled=enabled)


def autocast_context(enabled):
    amp_ns = getattr(torch, 'amp', None)
    if amp_ns is not None and hasattr(amp_ns, 'autocast'):
        return amp_ns.autocast('cuda', enabled=enabled)
    return getattr(torch.cuda, 'amp').autocast(enabled=enabled)


def metric_key(value):
    return f'{float(value):.6f}'


def results_summary_path(experiment_name):
    return Path(CFG['results_dir']) / f'{experiment_name}_summary.json'


def load_experiment_summary(experiment_name):
    path = results_summary_path(experiment_name)
    if not path.exists():
        return {}
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def save_experiment_summary(experiment_name, summary):
    path = results_summary_path(experiment_name)
    payload = json.dumps(summary, indent=2, sort_keys=True)
    path.write_text(payload + '\n', encoding='utf-8')
    print(f'Saved {experiment_name} summary -> {path}')


def train_one_run(model, train_loader, val_loader, loss_fn,
                  epochs, lr, device, run_name='run', verbose=True,
                  resume=None, use_amp=None, early_stopping_patience=None,
                  early_stopping_min_epochs=None, early_stopping_delta=None):
    """
    Full training + validation loop with optional AMP, early stopping, and resume.
    Saves the best checkpoint by val mIoU.
    Returns (history_dict, best_miou).
    """
    if resume is None:
        resume = CFG['resume_training']
    if use_amp is None:
        use_amp = CFG['use_amp'] and device == 'cuda'
    if early_stopping_patience is None:
        early_stopping_patience = CFG['early_stopping_patience']
    if early_stopping_min_epochs is None:
        early_stopping_min_epochs = CFG['early_stopping_min_epochs']
    if early_stopping_delta is None:
        early_stopping_delta = CFG['early_stopping_delta']

    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = make_grad_scaler(use_amp)
    history = {'train_loss': [], 'val_miou': [], 'val_pixel_acc': [], 'lr': []}
    best_miou = 0.0
    start_epoch = 0
    epochs_without_improvement = 0
    ckpt_path = os.path.join(CFG['checkpoints_dir'], f'best_{run_name}.pth')
    last_ckpt_path = os.path.join(CFG['checkpoints_dir'], f'last_{run_name}.pth')

    if resume and os.path.exists(last_ckpt_path):
        checkpoint = torch.load(last_ckpt_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if use_amp and checkpoint.get('scaler_state_dict'):
                scaler.load_state_dict(checkpoint['scaler_state_dict'])
            history = checkpoint.get('history', history)
            best_miou = checkpoint.get('best_miou', best_miou)
            start_epoch = checkpoint.get('epoch', -1) + 1
            epochs_without_improvement = checkpoint.get(
                'epochs_without_improvement', epochs_without_improvement
            )
            print(f'Resuming {run_name} from epoch {start_epoch + 1}/{epochs}.')

    for epoch in range(start_epoch, epochs):
        model.train()
        epoch_loss, n_batch = 0.0, 0
        for images, masks, point_masks in train_loader:
            images = images.to(device, non_blocking=(device == 'cuda'))
            masks = masks.to(device, non_blocking=(device == 'cuda'))
            point_masks = point_masks.to(device, non_blocking=(device == 'cuda'))
            optimizer.zero_grad(set_to_none=True)
            with autocast_context(use_amp):
                logits = model(images)
                loss = loss_fn(logits, masks, point_masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            n_batch += 1
        scheduler.step()

        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for images, masks, _ in val_loader:
                images = images.to(device, non_blocking=(device == 'cuda'))
                with autocast_context(use_amp):
                    preds = model(images).argmax(dim=1).cpu()
                all_preds.append(preds)
                all_targets.append(masks)
        m = compute_metrics(torch.cat(all_preds), torch.cat(all_targets), CFG['num_classes'])

        avg_loss = epoch_loss / max(n_batch, 1)
        history['train_loss'].append(avg_loss)
        history['val_miou'].append(m['miou'])
        history['val_pixel_acc'].append(m['pixel_acc'])
        history['lr'].append(scheduler.get_last_lr()[0])

        improved = m['miou'] > (best_miou + early_stopping_delta)
        if improved:
            best_miou = m['miou']
            epochs_without_improvement = 0
            torch.save({
                'epoch': epoch,
                'run_name': run_name,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict() if use_amp else None,
                'history': history,
                'best_miou': best_miou,
                'best_metrics': m,
                'epochs_without_improvement': epochs_without_improvement,
                'cfg': dict(CFG),
            }, ckpt_path)
        else:
            epochs_without_improvement += 1

        torch.save({
            'epoch': epoch,
            'run_name': run_name,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if use_amp else None,
            'history': history,
            'best_miou': best_miou,
            'last_metrics': m,
            'epochs_without_improvement': epochs_without_improvement,
            'cfg': dict(CFG),
        }, last_ckpt_path)

        if verbose:
            print(f'  [{run_name}] ep{epoch+1:02d}/{epochs} '
                  f'loss={avg_loss:.4f} mIoU={m["miou"]:.4f} px_acc={m["pixel_acc"]:.4f}')

        if (epoch + 1) >= early_stopping_min_epochs and epochs_without_improvement >= early_stopping_patience:
            print(f'  [{run_name}] Early stopping at epoch {epoch+1} after {epochs_without_improvement} stale epoch(s).')
            break

    print(f'  [{run_name}] Best mIoU={best_miou:.4f} saved -> {ckpt_path}')
    return history, best_miou

print('Training utilities ready.')

In [ ]:
# ============================================================
# Cell 7 — Experiment A: Point label density ablation (Effect of point label density on segmentation performance)
# ============================================================
# Purpose:   How does mIoU degrade as annotation becomes sparser?
# Hypothesis: Nonlinear degradation — robust above ~0.5%, sharp drop below it (rare classes absent from batches).
# Controlled: density in {0.001, 0.005, 0.01, 0.05}
# Fixed:      gamma = 2.0

required_symbols = [
    'train_one_run', 'build_model', 'PartialCrossEntropyLoss', 'PotsdamDataset',
    'load_experiment_summary', 'save_experiment_summary', 'metric_key'
]
missing_symbols = [name for name in required_symbols if name not in globals()]
if missing_symbols:
    raise RuntimeError(
        'Experiment dependencies are missing from the current kernel: ' + ', '.join(missing_symbols) + '. '
        'Rerun Cells 3, 4, 5, and 6, then run Cell 7 again.'
    )

print('=' * 60)
print('EXPERIMENT A: Point density vs. mIoU')
print('=' * 60)
FIXED_GAMMA_A = 2.0
EXPERIMENT_A_NAME = 'experiment_a_density'
selected_densities = list(CFG['point_densities'])
density_override = CFG.get('experiment_a_density_override')
if density_override is not None:
    selected_densities = [float(density_override)]
    print(f'Single-run override active. Only density={density_override*100:.3f}% will run.')

saved_summary = load_experiment_summary(EXPERIMENT_A_NAME)
saved_runs = saved_summary.get('runs', {})
density_results = {}

for density in selected_densities:
    key = metric_key(density)
    if key in saved_runs:
        density_results[density] = saved_runs[key]
        print(f'\n--- density={density*100:.3f}% | gamma={FIXED_GAMMA_A} ---')
        print(f'Skipping completed run from summary cache: best_mIoU={saved_runs[key]["best_miou"]:.4f}')

loader_kwargs = {
    'batch_size': CFG['batch_size'],
    'num_workers': CFG['num_workers'],
    'pin_memory': CFG['pin_memory'],
}
if CFG['num_workers'] > 0:
    loader_kwargs['persistent_workers'] = True

for density in selected_densities:
    if density in density_results:
        continue

    print(f'\n--- density={density*100:.3f}% | gamma={FIXED_GAMMA_A} ---')
    train_ds = PotsdamDataset(train_img_paths, train_mask_paths,
                               CFG['img_size'], density, augment=True)
    val_ds   = PotsdamDataset(val_img_paths, val_mask_paths,
                               CFG['img_size'], density, augment=False)
    tr_ld = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    va_ld = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    torch.manual_seed(SEED)
    model = build_model(CFG['num_classes'], CFG['backbone'])
    hist, best = train_one_run(
        model, tr_ld, va_ld, PartialCrossEntropyLoss(gamma=FIXED_GAMMA_A),
        CFG['epochs'], CFG['lr'], CFG['device'], f'expA_d{density}'
    )
    result = {
        'density': float(density),
        'gamma': float(FIXED_GAMMA_A),
        'best_miou': float(best),
        'best_pixel_acc': float(max(hist['val_pixel_acc']) if hist['val_pixel_acc'] else float('nan')),
        'history': {
            'train_loss': [float(v) for v in hist['train_loss']],
            'val_miou': [float(v) for v in hist['val_miou']],
            'val_pixel_acc': [float(v) for v in hist['val_pixel_acc']],
            'lr': [float(v) for v in hist['lr']],
        },
    }
    density_results[density] = result
    saved_runs[metric_key(density)] = result
    save_experiment_summary(
        EXPERIMENT_A_NAME,
        {
            'experiment': EXPERIMENT_A_NAME,
            'run_mode': CFG['run_mode'],
            'fixed_gamma': float(FIXED_GAMMA_A),
            'runs': saved_runs,
        },
    )

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
densities = selected_densities
colors = ['#534AB7', '#1D9E75', '#BA7517', '#D85A30']
if len(densities) > len(colors):
    colors = list(plt.get_cmap('viridis')(np.linspace(0.15, 0.85, len(densities))))
bests = [density_results[d]['best_miou'] for d in densities]

axes[0].plot([d * 100 for d in densities], bests, 'o-', color='#534AB7', lw=2, ms=8)
for d, m in zip(densities, bests):
    axes[0].annotate(f'{m:.3f}', (d * 100, m), xytext=(0, 8), textcoords='offset points',
                     ha='center', fontsize=9)
if len(densities) > 1:
    axes[0].set_xscale('log')
axes[0].set_xlabel('Label density (%)')
axes[0].set_ylabel('Best val mIoU')
axes[0].set_title('Exp A: Density vs. mIoU')
axes[0].grid(alpha=0.3)

for density, color in zip(densities, colors):
    axes[1].plot(density_results[density]['history']['val_miou'], color=color, lw=1.5, label=f'{density*100:.3f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val mIoU')
axes[1].set_title('Exp A: mIoU curves')
axes[1].legend(title='Density')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG["figures_dir"]}/experiment_a_density.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nExp A summary:')
for density, score in zip(densities, bests):
    print(f'  density={density*100:.3f}%  best_mIoU={score:.4f}')
print(f'Intermediate summary file: {results_summary_path(EXPERIMENT_A_NAME)}')

In [ ]:
# ============================================================
# Cell 8 — Experiment B: Focal loss gamma ablation (Effect of focal loss gamma on performance with sparse labels)
# ============================================================
# Purpose:   Does focal hard-example mining help with sparse labels?
# Hypothesis: gamma in [1.0, 2.0] optimal. gamma=0 worst because easy background pixels dominate the sparse gradient budget.
# Controlled: gamma in {0.0, 0.5, 1.0, 2.0}
# Fixed:      density = 0.01 (1%)

required_symbols = [
    'train_one_run', 'build_model', 'PartialCrossEntropyLoss', 'PotsdamDataset',
    'load_experiment_summary', 'save_experiment_summary', 'metric_key'
]
missing_symbols = [name for name in required_symbols if name not in globals()]
if missing_symbols:
    raise RuntimeError(
        'Experiment dependencies are missing from the current kernel: ' + ', '.join(missing_symbols) + '. '
        'Rerun Cells 3, 4, 5, and 6, then run Cell 8 again.'
    )

print('=' * 60)
print('EXPERIMENT B: Focal gamma vs. mIoU')
print('=' * 60)
FIXED_DENSITY_B = CFG['fixed_density']
EXPERIMENT_B_NAME = 'experiment_b_gamma'
selected_gammas = list(CFG['focal_gammas'])
gamma_override = CFG.get('experiment_b_gamma_override')
if gamma_override is not None:
    selected_gammas = [float(gamma_override)]
    print(f'Single-run override active. Only gamma={gamma_override} will run.')

saved_summary = load_experiment_summary(EXPERIMENT_B_NAME)
saved_runs = saved_summary.get('runs', {})
gamma_results = {}

for gamma in selected_gammas:
    key = metric_key(gamma)
    if key in saved_runs:
        gamma_results[gamma] = saved_runs[key]
        print(f'\n--- gamma={gamma} | density={FIXED_DENSITY_B*100:.3f}% ---')
        print(f'Skipping completed run from summary cache: best_mIoU={saved_runs[key]["best_miou"]:.4f}')

loader_kwargs = {
    'batch_size': CFG['batch_size'],
    'num_workers': CFG['num_workers'],
    'pin_memory': CFG['pin_memory'],
}
if CFG['num_workers'] > 0:
    loader_kwargs['persistent_workers'] = True

for gamma in selected_gammas:
    if gamma in gamma_results:
        continue

    print(f'\n--- gamma={gamma} | density={FIXED_DENSITY_B*100:.3f}% ---')
    train_ds = PotsdamDataset(train_img_paths, train_mask_paths,
                               CFG['img_size'], FIXED_DENSITY_B, augment=True)
    val_ds   = PotsdamDataset(val_img_paths, val_mask_paths,
                               CFG['img_size'], FIXED_DENSITY_B, augment=False)
    tr_ld = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    va_ld = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    torch.manual_seed(SEED)
    model = build_model(CFG['num_classes'], CFG['backbone'])
    hist, best = train_one_run(
        model, tr_ld, va_ld, PartialCrossEntropyLoss(gamma=gamma),
        CFG['epochs'], CFG['lr'], CFG['device'], f'expB_g{gamma}'
    )
    result = {
        'gamma': float(gamma),
        'density': float(FIXED_DENSITY_B),
        'best_miou': float(best),
        'best_pixel_acc': float(max(hist['val_pixel_acc']) if hist['val_pixel_acc'] else float('nan')),
        'history': {
            'train_loss': [float(v) for v in hist['train_loss']],
            'val_miou': [float(v) for v in hist['val_miou']],
            'val_pixel_acc': [float(v) for v in hist['val_pixel_acc']],
            'lr': [float(v) for v in hist['lr']],
        },
    }
    gamma_results[gamma] = result
    saved_runs[metric_key(gamma)] = result
    save_experiment_summary(
        EXPERIMENT_B_NAME,
        {
            'experiment': EXPERIMENT_B_NAME,
            'run_mode': CFG['run_mode'],
            'fixed_density': float(FIXED_DENSITY_B),
            'runs': saved_runs,
        },
    )

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gammas = selected_gammas
colors = ['#B4B2A9', '#9FE1CB', '#5DCAA5', '#0F6E56']
if len(gammas) > len(colors):
    colors = list(plt.get_cmap('viridis')(np.linspace(0.15, 0.85, len(gammas))))
gmiou = [gamma_results[g]['best_miou'] for g in gammas]

bars = axes[0].bar([f'gamma={g}' for g in gammas], gmiou,
                   color=colors, edgecolor='white', lw=0.5)
for bar, score in zip(bars, gmiou):
    axes[0].text(bar.get_x() + bar.get_width() / 2, score + 0.002, f'{score:.3f}',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_xlabel('Focal gamma')
axes[0].set_ylabel('Best val mIoU')
axes[0].set_title('Exp B: Gamma vs. mIoU')
axes[0].grid(axis='y', alpha=0.3)

for gamma, color in zip(gammas, colors):
    axes[1].plot(gamma_results[gamma]['history']['val_miou'], color=color, lw=1.8, label=f'g={gamma}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val mIoU')
axes[1].set_title('Exp B: mIoU curves')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG["figures_dir"]}/experiment_b_gamma.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nExp B summary:')
for gamma, score in zip(gammas, gmiou):
    print(f'  gamma={gamma}  best_mIoU={score:.4f}')
print(f'Intermediate summary file: {results_summary_path(EXPERIMENT_B_NAME)}')

In [ ]:
# ============================================================
# Cell 9 — Qualitative results visualization
# ============================================================

def class_to_rgb(class_map):
    rgb = np.zeros((*class_map.shape, 3), dtype=np.uint8)
    for i, c in enumerate(CLASS_COLORS_RGB):
        rgb[class_map == i] = c
    return rgb

def denormalize(t):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    return (t.permute(1,2,0).numpy() * std + mean).clip(0,1)

def visualize_predictions(model, dataset, device, n=4, seed=0):
    model.eval()
    rng = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), min(n, len(dataset)))
    fig, axes = plt.subplots(n, 4, figsize=(20, 5.5*n))
    if n == 1: axes = axes[np.newaxis,:]
    titles = ['Input','Point labels (red)','Prediction','Ground truth']
    for row, idx in enumerate(idxs):
        img_t, mask, pm = dataset[idx]
        with torch.no_grad():
            pred = model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
        img_np  = denormalize(img_t)
        pt_vis  = img_np.copy()
        pt_vis[pm.numpy() == 1] = [1.0, 0.0, 0.0]
        panels  = [img_np, pt_vis, class_to_rgb(pred)/255, class_to_rgb(mask.numpy())/255]
        for col, (panel, title) in enumerate(zip(panels, titles)):
            axes[row, col].imshow(panel); axes[row, col].axis('off')
            if row == 0: axes[row, col].set_title(title, fontsize=11)
    patches = [
        mpatches.Patch(
            color=(col[0] / 255.0, col[1] / 255.0, col[2] / 255.0),
            label=name,
        )
        for col, name in zip(CLASS_COLORS_RGB, CLASS_NAMES)
    ]
    fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=10, bbox_to_anchor=(0.5,-0.02))
    plt.suptitle('Qualitative results — Partial CE Loss segmentation', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{CFG["figures_dir"]}/qualitative_results.png', dpi=150, bbox_inches='tight')
    plt.show()

# Load best model from Experiment A
BEST_DENSITY = max(density_results, key=lambda d: density_results[d]['best_miou']) \
    if density_results else 0.01
ckpt = os.path.join(CFG['checkpoints_dir'], f'best_expA_d{BEST_DENSITY}.pth')
if os.path.exists(ckpt):
    bm = build_model(CFG['num_classes'], CFG['backbone'])
    checkpoint = torch.load(ckpt, map_location=CFG['device'])
    bm.load_state_dict(checkpoint_model_state(checkpoint))
    bm.to(CFG['device'])
    vis_ds = PotsdamDataset(val_img_paths, val_mask_paths, CFG['img_size'], BEST_DENSITY)
    visualize_predictions(bm, vis_ds, CFG['device'])
else:
    print(f'Checkpoint not found: {ckpt}. Run Experiment A first.')

In [ ]:
# ============================================================
# Cell 10 — Results summary table
# ============================================================

print('=' * 60)
print('FULL RESULTS SUMMARY')
print('=' * 60)

if density_results:
    print('\nExperiment A — Point Density (gamma=2.0)')
    print(f'  {"Density":>10} | {"Best mIoU":>10}')
    print('  ' + '-' * 25)
    for d in CFG['point_densities']:
        m = density_results.get(d, {}).get('best_miou', float('nan'))
        print(f'  {d*100:>9.1f}% | {m:>10.4f}')

if gamma_results:
    print(f'\nExperiment B — Focal Gamma (density={CFG["fixed_density"]*100:.1f}%)')
    print(f'  {"Gamma":>8} | {"Best mIoU":>10}')
    print('  ' + '-' * 22)
    for g in CFG['focal_gammas']:
        m = gamma_results.get(g, {}).get('best_miou', float('nan'))
        print(f'  {g:>8.1f} | {m:>10.4f}')

print('\nFigures saved to:', CFG['figures_dir'])
print('Checkpoints saved to:', CFG['checkpoints_dir'])
print('\nAll done!')
